<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. Validación Cruzada y KNN Regresor
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/07%20-%20Regression/Para%20Dummies/04_Seleccion_Modelos_Validacion_Cruzada_y_KNN_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del cuaderno 04 de Regresión, el último de este módulo. Si el cuaderno principal usó palabras como *k-Fold Cross-Validation*, *Data Leakage* o *k-NN Regressor* y sentiste que iba muy rápido, aquí vamos más despacio.

Al terminar podrás explicar, con tus propias palabras:
1. Por qué comparar modelos con una sola partición train/test puede ser engañoso.
2. Qué es la validación cruzada (k-Fold) y por qué es más confiable.
3. Cómo funciona k-NN, un modelo de regresión que no usa ninguna fórmula, sino "vecinos parecidos".
4. Cuándo conviene usar un modelo como la regresión lineal y cuándo uno como k-NN.

---
## 1. El problema de examinar una sola vez 🎲

Imagina que quieres decidir cuál de dos estudiantes es mejor en matemáticas, y para eso les haces **un solo examen**. Si por casualidad ese examen tenía justo las preguntas que el Estudiante A se sabía de memoria y las que el Estudiante B no repasó, tu conclusión puede ser completamente injusta — no porque A sea mejor en general, sino porque ese examen en particular le convino.

Lo mismo pasa cuando evaluamos un modelo con una sola partición de `train_test_split`: si por azar el conjunto de prueba quedó "fácil" o "difícil", la métrica que obtienes puede no reflejar qué tan bueno es realmente el modelo. Esto es especialmente riesgoso con pocos datos, donde cada partición aleatoria puede verse muy distinta a otra.

---
## 2. La solución: el examen rotativo (Validación Cruzada) 🔄

En vez de un solo examen, imagina que divides las preguntas en 5 grupos y haces 5 rondas: en cada ronda, un grupo de preguntas es el "examen" y los otros 4 grupos son con los que el estudiante repasó. Al final, promedias el resultado de las 5 rondas.

Eso es exactamente la **Validación Cruzada de k pliegues (k-Fold Cross-Validation)**: divides los datos en `k` partes (comúnmente 5 o 10), y por turnos cada parte actúa como "examen" mientras el modelo se entrena con las demás. El resultado final es el promedio de las `k` evaluaciones — mucho más confiable que un solo intento con una sola partición.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge

cal = fetch_california_housing(as_frame=True)
X = cal.data[['MedInc', 'HouseAge', 'AveRooms']][:500]
y = cal.target[:500]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_ridge = cross_val_score(Ridge(), X, y, cv=kf, scoring='r2')
scores_knn = cross_val_score(KNeighborsRegressor(n_neighbors=7), X, y, cv=kf, scoring='r2')

print(f'Ridge R² en 5 rondas: {scores_ridge.round(3)} | Promedio: {scores_ridge.mean():.3f}')
print(f'KNN R² en 5 rondas: {scores_knn.round(3)} | Promedio: {scores_knn.mean():.3f}')

### 🤔 ¿Qué acaba de pasar?

- `KFold(n_splits=5, ...)` define las reglas del "examen rotativo": 5 grupos, mezclados aleatoriamente (`shuffle=True`) para que cada grupo sea representativo.
- `cross_val_score(modelo, X, y, cv=kf, scoring='r2')` hace automáticamente las 5 rondas de entrenar-y-evaluar, y nos devuelve un R² por cada ronda.
- Fíjate que los 5 números de cada modelo no son idénticos — varían un poco de ronda en ronda, justo como le pasaría a un estudiante con distintos grupos de preguntas. El **promedio** de esos 5 números es una estimación mucho más confiable que cualquiera de ellos por separado.
- Comparamos dos modelos muy distintos (Ridge, una fórmula; KNN, "vecinos parecidos") con exactamente el mismo procedimiento, para que la comparación sea justa.

---
## 3. Predecir por parecido: k-Vecinos Más Cercanos (k-NN) 🏘️

Imagina que quieres estimar el precio de una casa y no tienes ninguna fórmula matemática, pero sí conoces los precios de venta reciente del vecindario. Una estrategia muy natural sería: "busco las 5 casas más parecidas a esta (mismo tamaño, misma zona, edad similar) y promedio sus precios de venta". Esa es literalmente la idea de **k-NN (k-Nearest Neighbors, o k-Vecinos Más Cercanos)**.

A diferencia de la regresión lineal, k-NN **no aprende ninguna fórmula ni ningún peso**: simplemente memoriza todos los datos de entrenamiento y, cuando le pides una predicción nueva, busca los `k` ejemplos más "cercanos" (parecidos) y promedia su resultado.

- Si `k` es muy pequeño (por ejemplo, `k=1`), la predicción depende de un solo vecino — muy sensible al ruido (riesgo de sobreajuste).
- Si `k` es muy grande, la predicción se vuelve un promedio demasiado general — pierde matices locales (riesgo de subajuste).

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler

# Tamaño (m2) y precio (millones COP) de casas ya vendidas
tamano = np.array([[40], [55], [60], [70], [85], [95], [110], [130]])
precio = np.array([120, 160, 175, 210, 250, 270, 320, 380])

# k-NN es sensible a la escala, así que escalamos primero
escalador = MinMaxScaler()
tamano_esc = escalador.fit_transform(tamano)

for k in [1, 3, 5]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(tamano_esc, precio)
    casa_nueva = escalador.transform([[75]])
    prediccion = knn.predict(casa_nueva)
    print(f"k={k} -> Precio estimado para una casa de 75 m2: {prediccion[0]:.1f} millones COP")

### 🤔 ¿Qué acaba de pasar?

- Creamos una tabla pequeña de casas ya vendidas, con su tamaño y precio.
- Escalamos el tamaño con `MinMaxScaler()` — un paso **obligatorio** en k-NN, porque el algoritmo mide distancias, y una variable con valores grandes dominaría injustamente el cálculo si no se escalan todas las variables a un rango comparable.
- Para una casa nueva de 75 m², probamos con `k=1`, `k=3` y `k=5`: nota cómo la predicción cambia según cuántos "vecinos" promediamos. Con `k=1` la predicción depende de una sola casa parecida; con `k=5` es un promedio más suavizado de varias casas cercanas en tamaño.

---
## 4. Dos formas muy distintas de predecir 🔀

| | Regresión Lineal | k-NN |
|---|---|---|
| ¿Cómo predice? | Con una fórmula fija (pesos aprendidos una sola vez). | Buscando los vecinos más parecidos en cada predicción nueva. |
| ¿Se puede explicar fácil? | Sí, con los coeficientes. | No tanto — es una "caja" basada en comparaciones. |
| ¿Capta curvas raras? | Solo si le agregas `PolynomialFeatures`. | Sí, de forma natural. |
| ¿Necesita escalar variables? | No es obligatorio. | Sí, siempre — porque mide distancias. |
| ¿Qué tan rápido predice? | Muy rápido (solo multiplica y suma). | Más lento con muchos datos (compara contra todos los vecinos). |

No hay un ganador absoluto: la elección depende de si necesitas explicar el modelo, de si sospechas relaciones curvas o complejas, y de cuántos datos tienes disponibles.

---
## 5. ¿La diferencia entre dos modelos es real, o pura suerte? 🪙

Si lanzas una moneda 10 veces y sale cara 6 veces, ¿dirías que la moneda está "cargada" hacia cara? Probablemente no — con tan pocos lanzamientos, 6 caras de 10 puede pasar fácilmente por puro azar. Si la lanzaras 10,000 veces y salieran 6,000 caras, ahí sí sospecharías que algo raro pasa con la moneda.

Comparar modelos tiene el mismo dilema: si el R² promedio de Ridge en las 5 rondas fue 0.62 y el de KNN fue 0.60, ¿esa diferencia de 0.02 es real o es "ruido" de la partición aleatoria? El cuaderno principal usa herramientas estadísticas formales (como la prueba $t$ y la prueba $F$) para responder justo esa pregunta con rigor. Aquí, con la intuición de la moneda, ya tienes la idea central: **una sola comparación nunca es suficiente evidencia — necesitas ver el patrón repetido varias veces (como hace la validación cruzada) antes de confiar en una conclusión.**

---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Hold-Out (una sola partición) | Puede dar una evaluación injusta si el conjunto de prueba fue "fácil" o "difícil" por azar. |
| Validación Cruzada (k-Fold) | Repite el examen `k` veces rotando qué datos son de prueba, y promedia los resultados. |
| `cross_val_score` | Automatiza las `k` rondas de entrenar y evaluar en una sola línea de código. |
| k-NN | Predice buscando los `k` ejemplos más parecidos y promediando su resultado — no usa ninguna fórmula fija. |
| Escalado en k-NN | Obligatorio, porque el algoritmo mide distancias entre variables. |
| Comparar modelos | Una diferencia pequeña entre dos modelos puede ser puro azar; hay que verla repetirse antes de confiar en ella. |

➡️ **Siguiente paso:** con esto concluyes la versión "Para Dummies" del módulo de Regresión. El siguiente módulo del curso es Clasificación, donde predecirás categorías en vez de números — puedes continuar en [00 - Introducción a la Clasificación (Para Dummies)](../../08%20-%20Classification/Para%20Dummies/00_Introduccion_Classification_Dummies.ipynb).

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
